In [9]:
# 원본데이터로드..
# 토크나이져 함수를 정의
    # 텍스트 전처리
    # 공백을 기준으로 단어단위로 분리
    # 영어는 전부 소문자로 변환
    # 어간 추출
    # 불용어 제거
# TFIDF를 정의
    # 토크나이져 매개변수 = 토크나이져 함수
    # ngram  (1,1)
# 파이프라인으로 tfidf, 머신러닝
# 파이프라인으로 학습
# 파이프라인으로 평가( classification_report)
# 과적합여부 확인
    # train 데이터와 test 데이터로 성능을 비교

In [10]:
# csv 데이터 호출
import pandas as pd
df = pd.read_csv('movie_data.csv')
df.head()

,review,target
0,Story of a man who has unnatural feelings for ...,0
1,Airport '77 starts as a brand new luxury 747 p...,0
2,This film lacked something I couldn't put my f...,0
3,"Sorry everyone,,, I know this is supposed to b...",0
4,When I was little my parents took me along to ...,0


In [11]:
df.shape

(1000, 2)

In [12]:
# 불용어 사전 다운로드
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [13]:
import re
from nltk.stem.porter import PorterStemmer
porter = PorterStemmer()
stops = stopwords.words('english')

def tokenizer(text):
    # 문서 토큰화 절차 1 : 불필요한 string 제거
    # 1. 영문, 공백, ., . 만 남기기
    clean = re.sub(r'[^A-Za-z\s.,]+','',text)
    # 2. 연속된 마침표(...)를 마침표 . 하나로
    clean = re.sub(r'\.{2.}', '.', clean)
    # 3. 연속된 공백 처리
    clean = re.sub(r'\s+', ' ', clean).strip()

    # 문서 토큰화 절차 2 : 영어는 전부 소문자로 변환
    clean = text.lower()

    # 문서 토큰화 절차 3 : 문서를 단어로 변환, 단어를 어간으로 변환, 불용어 제거
    stems = [porter.stem(word) for word in clean.split() if word not in stops]

    return stems

In [14]:
df['review'] = df['review'].apply(lambda x: tokenizer(str(x)))

In [15]:
df.head()

,review,target
0,"[stori, man, unnatur, feel, pig., start, open,...",0
1,"[airport, '77, start, brand, new, luxuri, 747,...",0
2,"[film, lack, someth, put, finger, first:, char...",0
3,"[sorri, everyone,,,, know, suppos, ""art"", film...",0
4,"[littl, parent, took, along, theater, see, int...",0


In [ ]:
# TfidfVectorizer() 도구 생성 및 설정
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(
    tokenizer=tokenizer,
    ngram_range=(1,1),
    token_pattern=None
    )

In [22]:
# 파이프라인 생성
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
pipeline = Pipeline([
    ('tfid',tfidf),
    ('clf',LogisticRegression())
])

In [26]:
# csv 데이터 재호출
import pandas as pd
df = pd.read_csv('movie_data.csv')
df.head()

,review,target
0,Story of a man who has unnatural feelings for ...,0
1,Airport '77 starts as a brand new luxury 747 p...,0
2,This film lacked something I couldn't put my f...,0
3,"Sorry everyone,,, I know this is supposed to b...",0
4,When I was little my parents took me along to ...,0


In [27]:
# 독립변수 / 종속변수 생성
X = df.review
y = df.target

# 훈련 / 테스트 데이터 분할
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)

In [28]:
# 파이프라인으로 모델 학습
pipeline.fit(x_train,y_train)

,steps,"[('tfid', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,<function tok...002554E2F3F60>


In [29]:
# 모델 성능 조회
pipeline.score(x_test,y_test)

0.92